In [1]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch

from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
from openretina.utils.plotting import (
    numpy_to_mp4_video,
)
from openretina.utils.file_utils import get_local_file_path
from openretina.utils.h5_handling import load_h5_into_dict
from openretina.models.core_readout import ViViTCoreReadout

from openretina.data_io.cyclers import LongCycler, ShortCycler
from openretina.data_io.hoefling_2024.dataloaders import natmov_dataloaders_v2
from openretina.data_io.hoefling_2024.responses import filter_responses, make_final_responses
from openretina.data_io.hoefling_2024.stimuli import movies_from_pickle
import os
import hydra

In [2]:
with hydra.initialize(config_path=os.path.join("..", "configs"), version_base="1.3"):
    cfg = hydra.compose(config_name="hoefling_2024_core_readout_low_res.yaml")

/home/bethge/bkr618/openretina_cache/notebook_example/tensorboard

In [3]:
your_chosen_root_folder = "/home/bethge/bkr618/openretina_cache"  # Change this with your desired path.

cfg.paths.cache_dir = your_chosen_root_folder

# We will also overwrite the output directory for the logs/model to the local folder.
cfg.paths.log_dir = your_chosen_root_folder
cfg.paths.output_dir = your_chosen_root_folder

os.environ["OPENRETINA_CACHE_DIRECTORY"] = your_chosen_root_folder

In [4]:
file_path = '/home/bethge/bkr618/openretina_cache/euler_lab/hoefling_2024/stimuli/rgc_natstim_72x64_joint_normalized_2024-10-11.pkl'
movie_stimuli = movies_from_pickle(file_path)

In [5]:
responses_path = "/home/bethge/bkr618/openretina_cache/data/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.h5"
responses_dict = load_h5_into_dict(file_path=responses_path)

filtered_responses_dict = filter_responses(responses_dict, **cfg.quality_checks)

final_responses = make_final_responses(filtered_responses_dict, response_type="natural")

Loading HDF5 file contents:   0%|          | 0/2077 [00:00<?, ?item/s]

Original dataset contains 7863 neurons over 67 fields
 ------------------------------------ 
Dropped 0 fields that did not contain the target cell types (67 remaining)
Overall, dropped 3034 neurons of non-target cell types (-38.59%).
 ------------------------------------ 
Dropped 0 fields with quality indices below threshold (67 remaining)
Overall, dropped 980 neurons over quality checks (-20.29%).
 ------------------------------------ 
Dropped 0 fields with classifier confidences below 0.25
Overall, dropped 705 neurons with classifier confidences below 0.25 (-18.32%).
 ------------------------------------ 
 ------------------------------------ 
Final dataset contains 3144 neurons over 67 fields
Total number of cells dropped: 4719 (-60.02%)


Upsampling natural spikes traces to get final responses.:   0%|          | 0/67 [00:00<?, ?it/s]

In [6]:
dataloaders = natmov_dataloaders_v2(
    neuron_data_dictionary=final_responses,
    movies_dictionary=movie_stimuli,
    allow_over_boundaries=True,
    batch_size=128,
    train_chunk_size=50,
    validation_clip_indices=cfg.dataloader.validation_clip_indices,
)

Creating movie dataloaders:   0%|          | 0/67 [00:00<?, ?it/s]

In [7]:
from openretina.data_io.base import compute_data_info
data_info = compute_data_info(neuron_data_dictionary=final_responses, movies_dictionary=movie_stimuli)


In [8]:
train_loader = LongCycler(dataloaders["train"])
val_loader = ShortCycler(dataloaders["validation"])

In [9]:
n_neurons_dict = data_info["n_neurons_dict"]

model = ViViTCoreReadout(
    input_shape=(128,2,50,72,64),
    n_neurons_dict=n_neurons_dict,
    channels = 2,
    Demb=64,  # Embedding dimension
    patch_size=12,  # Spatial patch size (H, W)
    temporal_patch_size=8,  # Temporal patch size
    num_spatial_blocks=3,  # Number of spatial transformer blocks
    num_temporal_blocks=2,  # Number of temporal transformer blocks
    num_heads=2,  # Number of attention heads
    mlp_ratio=4.0,  # MLP expansion ratio
    dropout=0.2,
    pad_frame=False,
    temporal_stride=1,
    spatial_stride=8,
    ptoken=0.2,  # Token dropout probability
    readout_bias=True,
    readout_init_mu_range=0.1,
    readout_init_sigma_range=0.18,
    readout_gamma=0.3,
    readout_reg_avg=False,
    learning_rate=5e-4,
    norm="rmsrnorm",
    patch_mode=1,
    pos_encoding=4,
    reg_tokens=20,
    ff_activation = "gelu",
    drop_path = 0.3,
    use_rope = True,
    use_causal_attention=False,
    reg_lambda = 1e-5,
    activation_lambda = 0,
    smooth_lambda = 1e-2

)
model = model.to('cuda')

/home/bethge/bkr618/open-retina/openretina/modules/readout/base.py:62: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


1. Creating Tokenizer...
2. Tokenizer created. Output shape: (43, 56, 64)
3. ViViT created with input shape: (43, 56, 64)
4. Spatial shape after patching: h=8, w=7
5. Core output shape: (64, 43, 8, 7)
[SparseAttentionViz] Initialized with outdir=/home/bethge/bkr618/openretina_cache/attn_sparse, n_layers=1, device=cuda, head_limit=None, target_session=None


In [10]:
from pytorch_lightning.utilities.model_summary import summarize


summary = summarize(model, max_depth=-1)  # full depth
print(summary)

    | Name                                                  | Type                               | Params | Mode 
-----------------------------------------------------------------------------------------------------------------------
0   | core                                                  | ViViTCoreWrapper                   | 396 K  | train
1   | core.tokenizer                                        | Tokenizer                          | 147 K  | train
2   | core.tokenizer.proj                                   | Conv3d                             | 147 K  | train
3   | core.tokenizer.norm                                   | LayerNorm                          | 128    | train
4   | core.tokenizer.spatial_pos_embedding                  | SinusoidalPosEmb                   | 0      | train
5   | core.tokenizer.spatial_pos_embedding.dropout          | Dropout                            | 0      | train
6   | core.tokenizer.temporal_pos_encoding                  | SinusoidalPosEmb    

In [11]:
import lightning

In [12]:
log_save_path = os.path.join(cfg.paths.output_dir, "notebook_example")
os.makedirs(log_save_path, exist_ok=True)

logger = lightning.pytorch.loggers.TensorBoardLogger(
    name="tensorboard/",
    save_dir=log_save_path,
)

In [13]:
early_stopping = lightning.pytorch.callbacks.EarlyStopping(
    monitor="val_correlation",
    patience=10,
    mode="max",
    verbose=False,
    min_delta=0.001,
)

lr_monitor = lightning.pytorch.callbacks.LearningRateMonitor(logging_interval="epoch")

model_checkpoint = lightning.pytorch.callbacks.ModelCheckpoint(
    monitor="val_correlation", mode="max", save_weights_only=False
)

In [14]:
from openretina.utils.transformer_utils import SparseAttentionViz

sparse_cb = SparseAttentionViz(
    outdir="/home/bethge/bkr618/openretina_cache/attn_sparse",
    n_layers=1,
    head_limit=1,
    device='cuda',
    target_session = 'session_3_ventral1_20201021'
)


[SparseAttentionViz] Initialized with outdir=/home/bethge/bkr618/openretina_cache/attn_sparse, n_layers=1, device=cuda, head_limit=1, target_session=session_3_ventral1_20201021


In [15]:
trainer = lightning.Trainer(max_epochs=100, logger=logger, callbacks=[early_stopping, lr_monitor, model_checkpoint], precision = '16-mixed') #add precision

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [16]:
trainer.fit(model, train_loader, val_loader)

You are using a CUDA device ('NVIDIA A100-PCIE-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/bethge/bkr618/.local/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name             | Type                               | Params | Mode 
--------------------------------------------------------------------------------
0 | core             | ViViTCoreWrapper                   | 396 K  | train
1 | readout          | MultiSampledGaussianReadoutWrapper | 223 K  | train
2 | loss             | PoissonLoss3d  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [18]:
test_loader = ShortCycler(dataloaders["test"])
trainer.test(model, dataloaders=[train_loader, val_loader, test_loader], ckpt_path="best")

Restoring states from the checkpoint path at /home/bethge/bkr618/openretina_cache/notebook_example/tensorboard/version_57/checkpoints/epoch=12-step=1742.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/bethge/bkr618/openretina_cache/notebook_example/tensorboard/version_57/checkpoints/epoch=12-step=1742.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃       DataLoader 1        ┃       DataLoader 2        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│     test_correlation      │    0.1609085351228714     │    0.16334302723407745    │    0.2587983012199402     │
│         test_loss         │     905.7183227539062     │    380.29827880859375     │     36.46363067626953     │
└───────────────────────────┴───────────────────────────┴───────────────────────────┴───────────────────────────┘

[{'test_loss/dataloader_idx_0': 905.7183227539062,
  'test_correlation/dataloader_idx_0': 0.1609085351228714},
 {'test_loss/dataloader_idx_1': 380.29827880859375,
  'test_correlation/dataloader_idx_1': 0.16334302723407745},
 {'test_loss/dataloader_idx_2': 36.46363067626953,
  'test_correlation/dataloader_idx_2': 0.2587983012199402}]

In [17]:
trainer.save_checkpoint("/home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example_reg.ckpt")

In [ ]:
import torch
import numpy as np
import os
import matplotlib.cm as cm

def extract_attention_maps(
    checkpoint_path,
    val_dataloader,
    target_session=None,
    outdir="./attention_viz",
    device='cuda',
    neuron_idx=100,
    head_idx=0,
    num_frames=30,
    batch_idx=None,
    center_frame=None
):
    """
    Extract attention maps from a trained model checkpoint.
    
    Args:
        checkpoint_path: Path to the .ckpt file
        val_dataloader: Validation dataloader (or list of dataloaders)
        target_session: Session name string to visualize (None for first)
        outdir: Output directory for saved arrays
        device: 'cuda' or 'cpu'
        neuron_idx: Which neuron to visualize (default: 100)
        head_idx: Which attention head to use (default: 0)
        num_frames: Number of frames to extract (default: 30)
        batch_idx: Specific batch index to use (None for random)
        center_frame: Specific center frame (None for random)
    
    Returns:
        dict with keys: 'original_frames', 'overlaid_frames', 'attention_maps', 'metadata'
    """
    
    # Load the checkpoint
    print(f"[AttentionExtractor] Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    # Load your model class and instantiate it
    # You'll need to replace this with your actual model class

    pl_module = ViViTCoreReadout.load_from_checkpoint(checkpoint_path, map_location=device)
    pl_module.eval()
    pl_module.to(device)
    
    print(f"[AttentionExtractor] Model loaded successfully")
    
    # Find the target session
    if not isinstance(val_dataloader, list):
        val_dataloader = [val_dataloader]
    
    session_name = None
    batch = None
    found = False
    
    for val_loader in val_dataloader:
        try:
            for item in val_loader:
                current_session_name = item[0]
                current_batch = item[1]
                
                if target_session is None or current_session_name == target_session:
                    session_name = current_session_name
                    batch = current_batch
                    found = True
                    print(f"[AttentionExtractor] Found session: {session_name}")
                    break
            
            if found:
                break
                
        except Exception as e:
            print(f"[AttentionExtractor] Error: {e}")
            continue
    
    if not found:
        raise ValueError(f"Could not find session '{target_session}'")
    
    # Extract frames
    frames = batch.inputs
    if not torch.is_tensor(frames) or frames.ndim != 5:
        raise ValueError(f"Expected 5D tensor, got {type(frames)} with shape {getattr(frames, 'shape', None)}")
    
    print(f"targets shape:{batch.targets.shape}")
    frames = frames.to(device)
    B, C, T, H0, W0 = frames.shape
    print(f"[AttentionExtractor] Frames shape: {frames.shape}")
    
    # Select batch index
    if batch_idx is None:
        b = torch.randint(0, B, ()).item()
    else:
        b = min(batch_idx, B - 1)
    
    # Find core
    core = _find_core(pl_module)
    core.eval()
    
    # Create output directory
    os.makedirs(outdir, exist_ok=True)
    
    with torch.no_grad():
        # Select center frame
        if center_frame is None:
            print("RANDOM FRAME")
            t0 = torch.randint(0, T, ()).item()
        else:
            t0 = min(center_frame, T - 1)
        
        # Calculate window
        t_start = max(0, t0 - num_frames // 2)
        t_end = min(T, t0 + (num_frames - num_frames // 2))
        
        window_len = t_end - t_start
        if window_len < num_frames:
            if t_start == 0:
                t_end = min(T, num_frames)
            else:
                t_start = max(0, T - num_frames)
            window_len = t_end - t_start
        
        if window_len < num_frames:
            print(f"[AttentionExtractor] Warning: Only {window_len} frames available (requested {num_frames})")
        
        print(f"[AttentionExtractor] Using frames [{t_start}:{t_end}) around center t0={t0}")
        
        # Tokenize only needed frames
        frames_window = frames[b:b+1, :, t_start:t_end]

        tokens = core.tokenizer(frames_window)

        # Extract attention
        attn_full = core.get_spatial_attention_maps(tokens, layer_idx=-1)
        if attn_full is None:
            raise ValueError("Attention maps returned None")
        
        # Get readout grid
        readout = pl_module.readout
        session_readout = readout[session_name]
        grid = session_readout.sample_grid(batch_size=1, sample=False)  # (1, 44, 1, 2)
        # Get original and tokenized dimensions
        H_orig, W_orig = H0,W0  # Original image dimensions
        # CAMBIA PER OTTENERE DYNAMICALLY
        H_tok, W_tok = 8,7  
        patch_size = 12
        spatial_stride = 8

        # Extract grid coordinates
        gx = grid[0, :, 0, 0]  # (44,) - x coordinates in tokenized space [-1, 1]
        gy = grid[0, :, 0, 1]  # (44,) - y coordinates in tokenized space [-1, 1]
        # Scale grid coordinates from tokenized space to original space
        # Calculate the effective scaling factor
        scale_factor_w = (W_tok - 1) / (W_orig - 1) if W_orig > 1 else 1
        scale_factor_h = (H_tok - 1) / (H_orig - 1) if H_orig > 1 else 1

        # Apply scaling to get coordinates in original space
        gx_orig = gx / scale_factor_w  # Scale x coordinates
        gy_orig = gy / scale_factor_h  # Scale y coordinates

        # Clamp to valid range
        gx_orig = torch.clamp(gx_orig, -1, 1)
        gy_orig = torch.clamp(gy_orig, -1, 1)

        # Convert from normalized [-1, 1] to pixel coordinates in tokenized space
        # This gives us which token/patch the neuron is attending to
        px = ((gx + 1) * 0.5 * (H_tok - 1)).round().long()
        py = ((gy + 1) * 0.5 * (W_tok - 1)).round().long()
        
        query_idx_per_neuron = py * W_tok + px
        
        # Extract attention maps for each frame
        original_frames = []
        overlaid_frames = []
        attention_maps = []
        
        num_attn_frames = attn_full.shape[0]
        for frame_idx in range(num_attn_frames):
            t = t_start + frame_idx
        #for frame_idx, t in enumerate(range(t_start, t_end)):
      
            attn_bt = attn_full[frame_idx]
            
            if head_idx >= attn_bt.shape[0]:
                raise ValueError(f"head_idx {head_idx} out of range (only {attn_bt.shape[0]} heads)")
            
            attn_head = attn_bt[head_idx]
            
            h_patch = core.new_h
            w_patch = core.new_w
            
            query_token = query_idx_per_neuron[neuron_idx].item()
            imp = attn_head[query_token].view(h_patch, w_patch)
  
            # Normalize
            imp = imp - imp.min()
            mx = imp.max()
            if mx > 0:
                imp = imp / mx
            
            # Upsample
            imp_up = torch.nn.functional.interpolate(
                imp[None, None, :, :],
                size=(H0, W0),
                mode="bilinear",
                align_corners=False
            ).squeeze().cpu().numpy()
            
            # Store attention map
            attention_maps.append(imp_up)
            
            # Get original frame
            frame_np = frames[b, :, t].cpu().numpy()
            base = frame_np[0]
            #original_frames.append(base)

            # Normalize base to 0-1 for proper overlay
            base_norm = (base - base.min()) / (base.max() - base.min() + 1e-8)

            # Create overlay
            hot_cmap = cm.get_cmap('viridis')
            imp_colored = hot_cmap(imp_up)[:, :, :3]

            alpha = 0.45
            if base.ndim == 2:
                base_rgb = np.stack([base_norm, base_norm, base_norm], axis=-1)
            else:
                # If your base is already 3-channel, also normalize
                base_rgb = (base - base.min()) / (base.max() - base.min() + 1e-8)

            # Blend
            overlaid = (1 - alpha) * base_rgb + alpha * imp_colored
            overlaid = np.clip(overlaid, 0, 1)
            overlaid_frames.append(overlaid)
            original_frames.append(frame_np)

        
        # Stack into arrays
        original_frames_arr = np.stack(original_frames, axis=0)
        overlaid_frames_arr = np.stack(overlaid_frames, axis=0)
        attention_maps_arr = np.stack(attention_maps, axis=0)
        
        # Save arrays
        save_name = f"{session_name}_b{b}_neuron{neuron_idx}_head{head_idx}"
        original_path = os.path.join(outdir, f"{save_name}_original.npy")
        overlaid_path = os.path.join(outdir, f"{save_name}_overlaid.npy")
        attention_path = os.path.join(outdir, f"{save_name}_attention.npy")
        
        np.save(original_path, original_frames_arr)
        np.save(overlaid_path, overlaid_frames_arr)
        np.save(attention_path, attention_maps_arr)
        print(f"save path:{overlaid_path}")
        print(f"[AttentionExtractor] Saved to {outdir}:")
        print(f"  - Original frames: {original_frames_arr.shape}")
        print(f"  - Overlaid frames: {overlaid_frames_arr.shape}")
        print(f"  - Attention maps: {attention_maps_arr.shape}")
        
        # Return results
        metadata = {
            'session_name': session_name,
            'batch_idx': b,
            'neuron_idx': neuron_idx,
            'head_idx': head_idx,
            'center_frame': t0,
            'frame_range': (t_start, t_end),
            'shape': (window_len, H0, W0)
        }
        
        return {
            'original_frames': original_frames_arr,
            'overlaid_frames': overlaid_frames_arr,
            'attention_maps': attention_maps_arr,
            'metadata': metadata
        }


def _find_core(pl_module):
    """Helper function to find the core module."""
    for name in ["core", "core_wrapper", "core_readout"]:
        if hasattr(pl_module, name):
            obj = getattr(pl_module, name)
            if hasattr(obj, "tokenizer") and hasattr(obj, "get_spatial_attention_maps"):
                print(f"[AttentionExtractor] Found core: {name}")
                return obj
    if hasattr(pl_module, "module"):
        return _find_core(pl_module.module)
    raise RuntimeError("No core with tokenizer+get_spatial_attention_maps found")



In [15]:
results = extract_attention_maps(
    checkpoint_path="/home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example.ckpt",
    val_dataloader=val_loader,
    target_session="session_3_ventral2_20210929",
    neuron_idx=5,
    head_idx=0,
    num_frames=150,
    batch_idx=9,  # Use first batch instead of random
    center_frame=10,  # Center around frame 50
    outdir="/home/bethge/bkr618/openretina_cache/attn_sparse")


[AttentionExtractor] Loading checkpoint from /home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example.ckpt
SONO IO:(64, 43, 8, 7)
1. Creating Tokenizer...
2. Tokenizer created. Output shape: (43, 56, 64)
3. ViViT created with input shape: (43, 56, 64)
4. Spatial shape after patching: h=8, w=7
5. Core output shape: (64, 43, 8, 7)
[SparseAttentionViz] Initialized with outdir=/home/bethge/bkr618/openretina_cache/attn_sparse, n_layers=1, device=cuda, head_limit=None, target_session=None
[AttentionExtractor] Model loaded successfully
[AttentionExtractor] Found session: session_3_ventral2_20210929
targets shape:torch.Size([15, 150, 31])
[AttentionExtractor] Frames shape: torch.Size([15, 2, 150, 72, 64])
[AttentionExtractor] Found core: core
[AttentionExtractor] Using frames [0:150) around center t0=10
tensor([[[[-0.0503, -0.0582]],

         [[ 0.0556, -0.0650]],

         [[ 0.0656, -0.0315]],

         [[ 0.0620, -0.0383]],

         [[-0.0540, -0.0156]],

         [[-0.0

/tmp/ipykernel_2887211/3897349656.py:223: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  hot_cmap = cm.get_cmap('viridis')


In [14]:
results = extract_attention_maps(
    checkpoint_path="/home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example.ckpt",
    val_dataloader=val_loader,
    target_session="session_5_ventral2_20210929",
    neuron_idx=10,
    head_idx=0,
    num_frames=150,
    batch_idx=9,  # Use first batch instead of random
    center_frame=10,  # Center around frame 50
    outdir="/home/bethge/bkr618/openretina_cache/attn_sparse")

[AttentionExtractor] Loading checkpoint from /home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example.ckpt
SONO IO:(64, 43, 8, 7)
1. Creating Tokenizer...
2. Tokenizer created. Output shape: (43, 56, 64)
3. ViViT created with input shape: (43, 56, 64)
4. Spatial shape after patching: h=8, w=7
5. Core output shape: (64, 43, 8, 7)
[SparseAttentionViz] Initialized with outdir=/home/bethge/bkr618/openretina_cache/attn_sparse, n_layers=1, device=cuda, head_limit=None, target_session=None
[AttentionExtractor] Model loaded successfully
[AttentionExtractor] Found session: session_5_ventral2_20210929
targets shape:torch.Size([15, 150, 44])
[AttentionExtractor] Frames shape: torch.Size([15, 2, 150, 72, 64])
[AttentionExtractor] Found core: core
[AttentionExtractor] Using frames [0:150) around center t0=10
tensor([[[[-6.2209e-03,  6.1676e-02]],

         [[-6.3552e-02,  2.7621e-02]],

         [[-1.5358e-02, -3.0951e-02]],

         [[ 5.7884e-02, -8.1940e-02]],

         [[-2.3

/tmp/ipykernel_2887211/3897349656.py:223: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  hot_cmap = cm.get_cmap('viridis')


In [44]:
import torch
from pytorch_lightning import LightningModule

# 1. Load model from checkpoint
model = ViViTCoreReadout.load_from_checkpoint("/home/bethge/bkr618/openretina_cache/model_checkpoints/transformer_example.ckpt")
model.eval()
model.freeze()

# 2. Prepare your new data loader
new_loader = test_loader

# 3. Run prediction
all_preds = []
all_targets = []  # optional, if you have labels
with torch.no_grad():
    for batch in new_loader:
        # adjust unpacking depending on your dataset
        x = batch[1].inputs
        preds = model(x)
        all_preds.append(preds.cpu())

        if len(batch) > 1:
            all_targets.append(batch[1].targets.cpu())

all_preds = torch.cat(all_preds)
if all_targets:
    all_targets = torch.cat(all_targets)


SONO IO:(64, 43, 8, 7)
1. Creating Tokenizer...
2. Tokenizer created. Output shape: (43, 56, 64)
3. ViViT created with input shape: (43, 56, 64)
4. Spatial shape after patching: h=8, w=7
5. Core output shape: (64, 43, 8, 7)
[SparseAttentionViz] Initialized with outdir=/home/bethge/bkr618/openretina_cache/attn_sparse, n_layers=1, device=cuda, head_limit=None, target_session=None


/home/bethge/bkr618/open-retina/openretina/modules/readout/base.py:62: UserWarning: Readout is NOT initialized with mean activity but with 0!
  warnings.warn("Readout is NOT initialized with mean activity but with 0!")


RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 80 but got size 42 for tensor number 1 in the list.

In [50]:
all_targets[2].shape

torch.Size([1, 750, 74])

In [14]:
print(torch.cuda.memory_allocated()/1e9)


0.652217344


In [10]:
# Clear cache first
torch.cuda.empty_cache()

# Check before
print("=== BEFORE MODEL ===")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Move model to GPU
model = model.to('cuda')

print("\n=== AFTER MODEL TO GPU ===")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")


=== BEFORE MODEL ===
Allocated: 0.00 GB
Reserved: 0.00 GB

=== AFTER MODEL TO GPU ===
Allocated: 0.00 GB
Reserved: 0.00 GB


In [21]:
# Create dummy batch
dummy_input = torch.randn(64, 2, 50, 72, 64).to('cuda')

print("\n=== AFTER CREATING INPUT ===")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")



=== AFTER CREATING INPUT ===
Allocated: 0.13 GB
Reserved: 0.14 GB


In [24]:
# Forward pass
output = model(dummy_input)

print("\n=== AFTER FORWARD PASS ===")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

OutOfMemoryError: CUDA out of memory. Tried to allocate 226.00 MiB. GPU 0 has a total capacity of 39.39 GiB of which 73.94 MiB is free. Including non-PyTorch memory, this process has 39.31 GiB memory in use. Of the allocated memory 38.73 GiB is allocated by PyTorch, and 94.88 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [20]:
import gc
del model, dummy_input, output  # etc.
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Backward pass
loss = output.sum()
loss.backward()

print("\n=== AFTER BACKWARD PASS ===")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

NameError: name 'output' is not defined

: 

In [ ]:
import gc
from tqdm import tqdm
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = VideoTokenizer(
    img_size=(72, 64),
    patch_size=(8, 8),
    temporal_patch_size=5,
    in_channels=2,
    Demb=128,
    ptoken=0.1
).to(device)

transformer = SpatioTemporalTransformer(
    Demb=128,
    num_spatial_blocks=4,
    num_temporal_blocks=4,
    num_heads=8,
    mlp_ratio=4.0,
    dropout=0.1,
    chunk_size=64
).to(device)

tokenizer.eval()
transformer.eval()

# dictionary for results
outputs_dict = {}

# loop with progress bar
for session_idx, item in enumerate(tqdm(train_loader, desc="Processing sessions", unit="session")):
    inputs = item[1].inputs.to(device)
    session_name = item[0]

    with torch.no_grad():
        embeddings, TP, SP = tokenizer(inputs)
        output = transformer(embeddings, TP, SP)
        output_cpu = output.cpu()

    outputs_dict[session_name] = output_cpu

    del inputs, embeddings, output
    torch.cuda.empty_cache()
    gc.collect()

    torch.cuda.synchronize()  # ensure memory freed before next iteration

# summary
print(f"\nProcessed {len(outputs_dict)} sessions total.")


Processing batches:   0%|          | 0/134 [00:00<?, ?batch/s]

Processing batches: 100%|██████████| 134/134 [01:31<00:00,  1.47batch/s]


Processed 67 batches total.


In [2]:
import gc
import os
import torch
from pathlib import Path
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Initialize the TransformerCoreWrapper
core = TransformerCoreWrapper(
    input_shape = (128, 2, 50, 72, 64),
    in_channels=2,
    img_size=(72, 64),
    patch_size=(8, 8),
    temporal_patch_size=5,
    emb_dim=128,
    ptoken=0.1,
    num_spatial_blocks=4,
    num_temporal_blocks=4,
    num_heads=8,
    mlp_ratio=4.0,
    dropout=0.1,
    chunk_size=64,
    gamma_weights=0.001,
    gamma_attention=0.01,
).to(device)

# Put in eval mode
core.eval()

dummy_input = torch.randn(128, 2, 50, 72, 64).to(device)  # (B, C, T, H, W)

with torch.no_grad():
    output = core(dummy_input)
    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    
del dummy_input, output
torch.cuda.empty_cache()
gc.collect()

Input shape: torch.Size([128, 2, 50, 72, 64])
Output shape: torch.Size([128, 128, 10, 9, 8])


177

In [16]:
import gc
import os
from pathlib import Path
from tqdm import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
viz_folder = Path("/home/bethge/bkr618/open-retina/core_visualizations")
os.makedirs(viz_folder, exist_ok=True)

core = TransformerCoreWrapper(
    input_shape=(128, 2, 50, 72, 64),
    in_channels=2,
    img_size=(72, 64),
    patch_size=(8, 8),
    temporal_patch_size=5,
    emb_dim=128,
    ptoken=0.1,
    num_spatial_blocks=4,
    num_temporal_blocks=4,
    num_heads=8,
    mlp_ratio=4.0,
    dropout=0.1,
    chunk_size=64,
    gamma_weights=0.001,
    gamma_attention=0.01,
).to(device)

core.eval()

outputs_dict = {}
VISUALIZE_EVERY = 20  # adjust frequency

for session_idx, item in enumerate(tqdm(train_loader, desc="Processing sessions", unit="session")):
    session_name = item[0]
    inputs = item[1].inputs.to(device, non_blocking=True)

    try:
        with torch.no_grad():
            
            output = core(inputs)
            # store CPU copy only
            outputs_dict[session_name] = output.detach().cpu()

    finally:
        # drop GPU refs
        del inputs
        if "output" in locals(): del output
        torch.cuda.empty_cache()
        gc.collect()
        if device == "cuda":
            torch.cuda.synchronize()

    if (session_idx + 1) % VISUALIZE_EVERY == 0:
        core.save_weight_visualizations(
            folder_path=str(viz_folder),
            file_format="png",
            state_suffix=f"_session_{session_idx+1}"
        )
        torch.cuda.empty_cache()
        gc.collect()
        if device == "cuda":
            torch.cuda.synchronize()

print(f"\nProcessed {len(outputs_dict)} sessions total.")



Processing sessions:   0%|          | 0/134 [00:00<?, ?session/s]

Processing sessions:  15%|█▍        | 20/134 [00:21<06:12,  3.26s/session]

Saved transformer visualizations to /home/bethge/bkr618/open-retina/core_visualizations/transformer_visualizations


Processing sessions:  29%|██▉       | 39/134 [00:34<01:07,  1.40session/s]

Saved transformer visualizations to /home/bethge/bkr618/open-retina/core_visualizations/transformer_visualizations


...

Saved transformer visualizations to /home/bethge/bkr618/open-retina/core_visualizations/transformer_visualizations


Processing sessions:  90%|████████▉ | 120/134 [02:26<00:50,  3.63s/session]

Saved transformer visualizations to /home/bethge/bkr618/open-retina/core_visualizations/transformer_visualizations


Processing sessions: 100%|██████████| 134/134 [02:39<00:00,  1.19s/session]


Processed 67 sessions total.


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

def extract_attn_for_batch(core, raw_frames, layer_idx=-1, head=None):
    """
    raw_frames: tensor (B, C, T, H0, W0) in the same format your forward expects.
    Returns:
        attn: tensor (B, T, num_heads, P, P)
    """
    core.eval()
    with torch.no_grad():
        # Tokenize exactly how forward does. Your tokenizer likely expects (B,C,T,H,W)
        tokens = core.tokenizer(raw_frames)            # (B, T, P, Demb)
        attn = core.get_spatial_attention_maps(tokens, layer_idx=layer_idx)  # (B*T, Hn, P, P)
        if attn is None:
            raise RuntimeError("No attention returned. Check layer_idx or get_spatial_attention_maps.")
        B, C, T, H0, W0 = raw_frames.shape
        bt, n_heads, P, _ = attn.shape
        assert bt == B * T
        attn = attn.view(B, T, n_heads, P, P)         # (B, T, Hn, P, P)
        if head is not None:
            attn = attn[:, :, head:head+1, :, :]      # keep one head
        return attn  # (B, T, Hs, P, P)

def attn_to_spatial_map(attn, new_h, new_w,
                        avg_heads=True,
                        agg="mean_query"):
    """
    Convert (B, T, Hs, P, P) --> (B, T, new_h, new_w) spatial importance maps.
    agg: "mean_query" (average attention over query dimension) or
         "select_query" (you provide an index) or
         "attn_to_query" (sum attention of a single query -> shows where that query attends).
    By default we average queries to get global key-importance.
    """
    B, T, Hs, P, _ = attn.shape
    assert P == new_h * new_w

    # Option 1: average heads then average queries -> global importance per key patch
    if avg_heads:
        attn_h = attn.mean(dim=2)    # (B,T,P,P)
    else:
        # keep heads dimension
        attn_h = attn  # (B,T,Hs,P,P) - not supported by the rest of this simple pipeline
        raise NotImplementedError("Non-averaged-heads not implemented in this helper.")

    if agg == "mean_query":
        # for each batch/time, compute importance of each key patch by averaging over queries
        # attn_h[b,t,q,k] -> importance_k = mean_q attn_h[b,t,q,k]
        importance = attn_h.mean(dim=2)         # (B, T, P)
    elif agg == "sum_query":
        importance = attn_h.sum(dim=2)
    else:
        raise ValueError("agg must be 'mean_query' or 'sum_query'")

    # reshape to patch grid
    importance = importance.view(B, T, new_h, new_w)  # (B, T, h, w)

    # normalize per map to [0,1]
    importance = importance - importance.amin(dim=(2,3), keepdim=True)
    denom = importance.amax(dim=(2,3), keepdim=True)
    denom[denom == 0] = 1.0
    importance = importance / denom

    return importance  # (B, T, new_h, new_w)

def upsample_to_frame(importance, H0, W0, mode="bilinear"):
    """
    importance: (B, T, h, w) float tensor in [0,1]
    returns: (B, T, H0, W0)
    """
    B, T, h, w = importance.shape
    x = importance.view(B*T, 1, h, w)
    up = F.interpolate(x, size=(H0, W0), mode=mode, align_corners=False)
    up = up.view(B, T, H0, W0)
    return up

def overlay_and_plot(raw_frames, heatmaps, idx_batch=0, idx_time=0, alpha=0.45, cmap='jet'):
    """
    raw_frames: (B, C, T, H0, W0) torch tensor in range [0,1] or similar
    heatmaps: (B, T, H0, W0) in [0,1]
    idx_batch, idx_time: which frame to show
    """
    img = raw_frames[idx_batch, :, idx_time].cpu().numpy()  # (C, H, W)
    heat = heatmaps[idx_batch, idx_time].cpu().numpy()      # (H, W)

    # convert C,H,W -> H,W,C for plt
    if img.shape[0] == 1:
        img_disp = img[0]
        plt.imshow(img_disp, cmap='gray', vmin=0, vmax=1)
    else:
        img_disp = np.transpose(img, (1,2,0))
        # if in [0,1] show as is; otherwise normalize
        plt.imshow(np.clip(img_disp, 0, 1))

    plt.imshow(heat, cmap=cmap, alpha=alpha, vmin=0, vmax=1)
    plt.axis('off')
    plt.title(f"batch {idx_batch} time {idx_time}")
    plt.show()

# --- Usage example in one cell ---
# raw_frames: (B, C, T, H0, W0) torch.tensor from your dataloader, values scaled to [0,1]
# core: instance of ViViTCoreWrapper

layer_idx = -1
attn = extract_attn_for_batch(core, raw_frames, layer_idx=layer_idx)   # (B, T, Hs, P, P)
importance = attn_to_spatial_map(attn, core.new_h, core.new_w, avg_heads=True, agg="mean_query")  # (B,T,h,w)
H0, W0 = raw_frames.shape[-2], raw_frames.shape[-1]
heat_upsampled = upsample_to_frame(importance, H0, W0)  # (B,T,H0,W0)

# plot one example
overlay_and_plot(raw_frames, heat_upsampled, idx_batch=0, idx_time=0, alpha=0.5)
